# LGD severity analysis

Notebook version of `src/model/lgd_severity_analysis.py`: same population, same numbers, one question per cell, short outputs.

**Population:** every charged-off loan (`LoanStatus == CHGOFF`), no 60-month seasoning filter. The filter fixes a PD-only problem (loans that haven't had time to default yet); it doesn't apply to a loan whose loss is already known.

**What "realized LGD" means here:** `GrossChargeOffAmount / GrossApproval`. Gross only: the data has no recovery amount.

Nothing in this notebook writes to the repo. The chart at the bottom is shown inline; the saved version is `outputs/figures/lgd/01_lgd_severity_by_segment.png` (made by the script).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for p in [ROOT, ROOT / "src" / "features", ROOT / "src" / "model"]:
    sys.path.insert(0, str(p))

import pandas as pd
import matplotlib.pyplot as plt
from lgd_severity_analysis import build_lgd_population   # same function the script uses
%matplotlib inline

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 60)

df = build_lgd_population()
print(f"{len(df):,} charged-off loans")

## 1. How is realized LGD spread out?

In [ ]:
lgd = df["realized_lgd"]
pd.DataFrame({
    "value": [lgd.mean(), lgd.median(), lgd.quantile(.05), lgd.quantile(.25), lgd.quantile(.75), lgd.quantile(.95)],
}, index=["mean", "median", "5th pct", "25th pct", "75th pct", "95th pct"]).round(3).T

Rows that look odd (loss above 100% of the loan, below zero, or no loss amount):

In [ ]:
pd.DataFrame({
    "loans": [(lgd > 1).sum(), (lgd < 0).sum(), df["GrossChargeOffAmount"].isna().sum()],
    "% of all": [(lgd > 1).mean() * 100, (lgd < 0).mean() * 100, df["GrossChargeOffAmount"].isna().mean() * 100],
}, index=["LGD above 1", "LGD below 0", "missing charge-off amount"]).round(2)

## 2. Collateral: how many loans, and does severity differ?

In [ ]:
g = df.groupby("CollateralInd", observed=True)["realized_lgd"].agg(loans="size", mean="mean", median="median")
g["% of loans"] = g["loans"] / g["loans"].sum() * 100
g.round(3)

## 3. Sector

In [ ]:
df.groupby("NaicsSectorName", observed=True)["realized_lgd"].agg(loans="size", mean="mean", median="median").sort_values("mean").round(3)

## 4. Loan size (five equal-sized groups)

In [ ]:
df["loan_size_bucket"] = pd.qcut(df["GrossApproval"], q=5, duplicates="drop")
df.groupby("loan_size_bucket", observed=True)["realized_lgd"].agg(loans="size", mean="mean", median="median").round(3)

## 5. Loan term

In [ ]:
df["term_bucket"] = pd.cut(df["TermInMonths"], bins=[0, 36, 60, 84, 120, 180, 300, 500])
df.groupby("term_bucket", observed=True)["realized_lgd"].agg(loans="size", mean="mean", median="median").round(3)

## 6. The four views as one chart

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

axes[0, 0].hist(df["realized_lgd"].clip(upper=1.5), bins=60, color="#4C72B0")
axes[0, 0].set_title("Realized LGD distribution (gross severity)")
axes[0, 0].set_xlabel("realized_lgd (clipped at 1.5 for display)")

coll = df.groupby("CollateralInd", observed=True)["realized_lgd"].mean()
axes[0, 1].bar(coll.index.astype(str), coll.values, color="#DD8452")
axes[0, 1].set_title("Mean realized LGD by CollateralInd")
axes[0, 1].set_ylabel("mean realized_lgd")

sector = df.groupby("NaicsSectorName", observed=True)["realized_lgd"].mean().sort_values()
axes[1, 0].barh(sector.index, sector.values, color="#55A868")
axes[1, 0].set_title("Mean realized LGD by sector")
axes[1, 0].set_xlabel("mean realized_lgd")
axes[1, 0].tick_params(axis="y", labelsize=7)

term = df.groupby("term_bucket", observed=True)["realized_lgd"].mean()
axes[1, 1].bar(term.index.astype(str), term.values, color="#8172B2")
axes[1, 1].set_title("Mean realized LGD by term bucket")
axes[1, 1].set_xlabel("term bucket (months)")
axes[1, 1].tick_params(axis="x", rotation=45)

fig.suptitle("LGD severity analysis: gross realized LGD by segment")
fig.tight_layout()
plt.show()